# Exhaustive Langfuse Prompt Management


In [ ]:
# 1. Imports: We import the Langfuse client and LangChain components.
# The CallbackHandler is crucial as it sends tracing data back to Langfuse so you can monitor your app.
from langfuse import Langfuse 
from langchain.chat_models import ChatOpenAI
from langfuse.langchain import CallbackHandler


In [ ]:
# 2. Initialisation: We initialise the Langfuse client. 
# This typically reads your public/secret keys from environment variables (LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY).
langfuse_client = Langfuse()


In [ ]:
# 3. Creating Prompts in Langfuse:
# Instead of hardcoding prompts in your code, you can store them in Langfuse.
# This allows you to update prompts without redeploying your application.

# a) Text Prompt: A simple string template with variables in curly braces {}.
langfuse_client.create_prompt(
    name="jokes-generator-text",
    type="text",
    prompt="tell me a joke in this {topic}",
    labels="production", # Labelling helps in version control (e.g., production, staging)
    config={
        "model": "gpt-3.5-turbo",
        "temperature": "1"
    }
)

# b) Chat Prompt: A list of messages (system, user, assistant). This is preferred for chat models.
langfuse_client.create_prompt(
    name="jokes-generator-chat",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "generate a joke best on the user provided topic"
        },
        {
            "role": "user",
            "content": "provide the topic {topic}"
        }
    ],
    labels="production",
    config={
        "model": "gpt-3.5-turbo",
        "temperature": "1"
    }
)


In [ ]:
# 4. Fetching Prompts:
# We fetch the 'production' version of our chat prompt from Langfuse.
langfuse_prompt = langfuse_client.get_prompt("jokes-generator-chat")


In [ ]:
# 5. LangChain Integration:
# Langfuse prompts can be converted directly into LangChain prompt templates.
prompt = langfuse_prompt.get_langchain_prompt()

# We extract the model configuration we stored in Langfuse.
model_name = langfuse_prompt.config.get("model", "gpt-3.5-turbo")
temperature_set = langfuse_prompt.config.get("temperature", "1")

# We initialise the ChatOpenAI model using the fetched configuration.
# llm = ChatOpenAI(model=model_name, temperature=temperature_set)


In [ ]:
# 6. LCEL (LangChain Expression Language):
# The '|' operator is used to chain components. Here we chain the prompt and the LLM.
# The output of the prompt becomes the input of the LLM.
# chain = prompt | llm


In [ ]:
# 7. Tracing:
# We create a callback handler. When passed to the chain, it logs the execution details to Langfuse.
handler = CallbackHandler()

# 8. Invocation:
# We invoke the chain with the required variable 'topic'.
# The config parameter is used to attach our tracing handler.
# response = chain.invoke(
#     {"topic": "cats"},
#     config={"callbacks": [handler]}
# )
# print("Langfuse Output:", response.content)


# --- PRACTICE TASK ---
# 1. Go to your Langfuse dashboard and view the trace for this execution.
# 2. Try creating a new prompt version in Langfuse UI (e.g., change the system message) and see how the output changes without modifying this code.
# 3. Try fetching the "jokes-generator-text" prompt instead and see how it works.
